In [8]:
import numpy as np
print("This program is just to see whether we can genrate circuit just before merging step in algorithm 1 \n using Python SDK\n input S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]] ")
S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]]
dif_qubits = []
dif_values = []
T = S
# Finding qubit with maximum difference between set T_0 and T_1 
# This is used in step 5

def find_qubit_with_unequal_sets(T):
    n = len(T[0])  # Number of qubits

    best_qubit = None
    Max_difference = float('-inf')  # Initialize to negative infinity

    for b in range(n):
        # Split T into T_0 and T_1 based on qubit b
        T_0 = [x for x in T if x[b] == 0]
        T_1 = [x for x in T if x[b] == 1]

        # Check if both sets are non-empty
        if len(T_0) != 0 and len(T_1) != 0:
            difference = abs(len(T_0) - len(T_1))
            if difference > Max_difference:
                Max_difference = difference
                best_qubit = b
                best_T_0 = T_0
                best_T_1 = T_1

    return best_qubit,best_T_0, best_T_1
P = find_qubit_with_unequal_sets(T)
# Step 5: Main loop
while len(T) > 1:
    # Step 6: Find the qubit b
    P = find_qubit_with_unequal_sets(T)  # We already implement this logic to find the best qubit best_T_0 and best_T_1 above
    b = P[0]
    T_0 = P[1]
    T_1 = P[2]
    
    # Step 7: Append b to dif_qubits
    
    dif_qubits.append(b)
    
    if len(T_0) < len(T_1):
        # Step 9: Set T = T_0 and append 0 to dif_values
        T = T_0
        dif_values.append(0)
    else:
        # Step 10: Set T = T_1 and append 1 to dif_values
        T = T_1
        dif_values.append(1)


# Step 14: Pop the last value appended to dif_qubits and store it as dif
dif = dif_qubits.pop();
print("dif is  ", dif, "\n i.e. our 3rd qubit line will act as dif since numbering started from 0 ")
# Step 15: Pop the last value that was appended to dif_values
dif_values.pop();
# Step 16: Store the single element in T as x_1
x_1 = T[0]
print("x1 is  ", x_1)
# step 17
# T_prime subset of S denote the set of strings that have the values in dif_values on the bits dif_qubits
T_prime = [x for x in S if all(x[q] == v for q, v in zip(dif_qubits, dif_values))]
# Step 18: Remove x_1 from T'
T_prime.remove(x_1)
# Step 19: Second While loop for T_prime
while len(T_prime) > 1:
    # Step 22: Find the qubit b_prime
    R = find_qubit_with_unequal_sets(T_prime)  # Implement logic to find the best qubit
    b_prime = R[0]
    T_prime_0 = R[1]
    T_prime_1 = R[2]
    
    # Step 7: Append b to dif_qubits
    
    dif_qubits.append(b_prime)
    
    if len(T_prime_0) < len(T_prime_1):
        # Step 9: Set T = T_0 and append 0 to dif_values
        T_prime = T_prime_0
        dif_values.append(0)
    else:
        # Step 10: Set T = T_1 and append 1 to dif_values
        T_prime = T_prime_1
        dif_values.append(1)
x_2 = T_prime[0]
print("x2 is ",x_2)
from classiq import Model, synthesize, QReg,show
from classiq.builtin_functions import RZGate,StatePreparation,CXGate,XGate
model = Model()
probabilities = [0, 0, 0, 0,0.25 , 0.25, 0, 0.25,0,0,0.25,0,0,0,0,0]
sp = StatePreparation(
    probabilities=probabilities, error_metric={"KL": {"upper_bound": 0.1}}
)
t=model.StatePreparation(params=sp)
x_params= XGate()
cx_params=["cx1","cx2","cx3","cx4"]
cx_out=["co1","co2","co3","co4"]
l=1
l1=[1,1,1,1]
u=0
# u will be used for numbering of CX Gates
if x_1[dif] != 1:
    t_out_dif= model.XGate(x_params, in_wires={"TARGET":t["OUT"][dif]})
    l=0
# Step 32: For qubit b in {1, 2, ..., n} \ dif
for b in range(0,4):
    if b != dif and x_1[b] != x_2[b]:
        cx_params[u] = CXGate()
        l1[b]=0
        # Step 34: Add a CNOT gate targeting b controlled on dif
        if l==0:
            cx_out[b]=model.CXGate(cx_params[u],in_wires={"TARGET":t["OUT"][b],"CTRL":t_out_dif["TARGET"]})            
        else:
            cx_out[b]=model.CXGate(cx_params[u],in_wires={"TARGET":t["OUT"][b],"CTRL":t["OUT"][dif]}) 
        u=u+1
x_last=["x1","x2","x3","x4"]
x_last_out=["xl1","xl2","xl3","xl4"]
# Step 37: For qubit b in dif_qubits
v=0
# v will be used for numbering of CX Gates
for b in dif_qubits:
    if x_2[b] != 1:
        x_last[i] = XGate()
        # Step 39: Add a NOT gate on line b
        if l1[b]==0:
            x_last_out[b]=model.XGate(x_last[v],in_wires={"TARGET":cx_out[b]["TARGET"]})
        else:
            x_last_out[b]=model.XGate(x_last[v],in_wires={"TARGET":t["OUT"][b]})
        v=v+1
quantum_program = synthesize(model.get_model())
show(quantum_program)         

This program is just to see whether we can genrate circuit just before merging step in algorithm 1 
 using Python SDK
 input S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]] 
dif is   2 
 i.e. our 3rd qubit line will act as dif since numbering started from 0 
x1 is   [1, 0, 1, 1]
x2 is  [1, 0, 0, 0]
Opening: https://platform.classiq.io/circuit/788f19fd-e7d6-42d8-8621-76a2aaa8d6c8?version=0.29.2


In [20]:
import numpy as np
from classiq import Model, synthesize, show
from classiq.builtin_functions import UnitaryGate
import sympy as sp

#Define the symbolic variables
omega, alpha, beta = sp.symbols('omega alpha beta', real=True)
alpha = 0 # Define your own alpha
beta = 1 # Define your own beta
# Define the G operator
G = sp.Matrix([[sp.sin(omega), sp.exp(sp.I * alpha) * sp.cos(omega)],
              [sp.exp(-sp.I * alpha) * sp.cos(omega), -sp.sin(omega)]])

# Define the column vectors |phi> and |0>
phi = sp.Matrix([alpha, beta])
zero_vector = sp.Matrix([1, 0])

# Define the equation G |phi> = |0>
equation = G * phi - zero_vector

# Solve for omega
solutions = sp.solve(equation, omega)
omega = solutions[0][0]

num_gate_qubits = 1
thetas = [np.pi / 3]
phis = [np.pi / 4]
pairs = [[0, 1]]
gate_matrix = np.identity(2**num_gate_qubits, dtype=complex)
for theta, phi, index in zip(thetas, phis, pairs):
    data_next = np.identity(2**num_gate_qubits, dtype=complex)
    data_next[index[0], index[0]] = 0
    data_next[index[1], index[1]] = 0
    data_next[np.ix_(index, index)] = [
        [
             sp.sin(omega) ,    sp.exp(sp.I * alpha) * sp.cos(omega)                                                                  
                                                                                    
        ],
        [sp.exp(-sp.I * alpha) * sp.cos(omega), -sp.sin(omega)]]
    gate_matrix = np.dot(gate_matrix, data_next)
gate_matrix = gate_matrix.tolist()

model = Model()
unitary_gate_params = UnitaryGate(data=gate_matrix)
model.UnitaryGate(unitary_gate_params)

quantum_program = synthesize(model.get_model())
show(quantum_program)


Opening: https://platform.classiq.io/circuit/1af40886-f5e5-4887-9c78-ebbd3ed4b0b6?version=0.29.2


In [14]:
from classiq import Model, QReg, synthesize, show
from classiq.builtin_functions import Mcx, StatePreparation

model = Model()

probabilities = (0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125)
sp_params = StatePreparation(probabilities=probabilities)
x = model.StatePreparation(sp_params)["OUT"]

mcx_params = Mcx(num_ctrl_qubits=len(x), ctrl_state="011")
model.Mcx(mcx_params, in_wires={"CTRL_IN": x})

quantum_program = synthesize(model.get_model())
show(quantum_program)


Opening: https://platform.classiq.io/circuit/07c5fe37-8db6-4788-b164-de27d79c88af?version=0.29.2


In [18]:
from classiq.builtin_functions import Mcx
from classiq import Model, synthesize

model = Model()
params = Mcx(
    num_qubits=4
)
print(model.QFT(params))

ValidationError: 1 validation error for Mcx
num_ctrl_qubits
  Must have control qubits (type=value_error)

In [4]:
from classiq import Model, RegisterUserInput, ControlState
from classiq.builtin_functions import Adder, Multiplier
from classiq import Model, QReg, synthesize, show
from classiq.builtin_functions import Mcx, StatePreparation
left_arg = RegisterUserInput(size=3)
right_arg = RegisterUserInput(size=3)
sum_arg = RegisterUserInput(size=4)
adder_params = Adder(left_arg=left_arg, right_arg=right_arg, inplace_arg="right")
multiplier_params = Multiplier(left_arg=left_arg, right_arg=sum_arg)

model = Model()
control_states = [
    ControlState(ctrl_state="1", name="first_control"),
    ControlState(num_ctrl_qubits=1, name="second_control"),
]
adder_out = model.Adder(params=adder_params, control_states=control_states)
multiplier_in = {"left_arg": adder_out["left_arg"], "right_arg": adder_out["sum"]}
model.Multiplier(
    params=multiplier_params, control_states=control_states, in_wires=multiplier_in
)
quantum_program = synthesize(model.get_model())
show(quantum_program)

Opening: https://platform.classiq.io/circuit/c37bdc41-5863-4975-901f-720c6ab59b76?version=0.29.2


In [27]:
import numpy as np
from classiq import Model, synthesize, show
from classiq.builtin_functions import UnitaryGate
alpha = 0.8 # Define your own alpha
beta = 0.6 # Define your own beta

# Define the column vectors |phi> and |0>

num_gate_qubits = 1
gate_matrix = np.identity(2**num_gate_qubits, dtype=complex)

data_next = np.identity(2**num_gate_qubits, dtype=complex)
data_next[np.ix_([0,1],[0,1])] = [
        [
             alpha ,    beta                                                                                                                                                     
        ],
        [-beta, alpha]]
gate_matrix = np.dot(gate_matrix, data_next)
gate_matrix = gate_matrix.tolist()
model = Model()
unitary_gate_params = UnitaryGate(data=gate_matrix)
model.UnitaryGate(unitary_gate_params)

quantum_program = synthesize(model.get_model())
show(quantum_program)

Opening: https://platform.classiq.io/circuit/d5d19d2c-ff1c-42dd-ab04-585c625a22cb?version=0.29.2


In [9]:
import numpy as np
print("This program is just to see whether we can genrate circuit just before merging step in algorithm 1 \n using Python SDK\n input S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]] ")
S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]]
dif_qubits = []
dif_values = []
T = S
# Finding qubit with maximum difference between set T_0 and T_1 
# This is used in step 5

def find_qubit_with_unequal_sets(T):
    n = len(T[0])  # Number of qubits

    best_qubit = None
    Max_difference = float('-inf')  # Initialize to negative infinity

    for b in range(n):
        # Split T into T_0 and T_1 based on qubit b
        T_0 = [x for x in T if x[b] == 0]
        T_1 = [x for x in T if x[b] == 1]

        # Check if both sets are non-empty
        if len(T_0) != 0 and len(T_1) != 0:
            difference = abs(len(T_0) - len(T_1))
            if difference > Max_difference:
                Max_difference = difference
                best_qubit = b
                best_T_0 = T_0
                best_T_1 = T_1

    return best_qubit,best_T_0, best_T_1
P = find_qubit_with_unequal_sets(T)
# Step 5: Main loop
while len(T) > 1:
    # Step 6: Find the qubit b
    P = find_qubit_with_unequal_sets(T)  # We already implement this logic to find the best qubit best_T_0 and best_T_1 above
    b = P[0]
    T_0 = P[1]
    T_1 = P[2]
    
    # Step 7: Append b to dif_qubits
    
    dif_qubits.append(b)
    
    if len(T_0) < len(T_1):
        # Step 9: Set T = T_0 and append 0 to dif_values
        T = T_0
        dif_values.append(0)
    else:
        # Step 10: Set T = T_1 and append 1 to dif_values
        T = T_1
        dif_values.append(1)


# Step 14: Pop the last value appended to dif_qubits and store it as dif
dif = dif_qubits.pop();
print("dif is  ", dif, "\n i.e. our 3rd qubit line will act as dif since numbering started from 0 ")
# Step 15: Pop the last value that was appended to dif_values
dif_values.pop();
# Step 16: Store the single element in T as x_1
x_1 = T[0]
print("x1 is  ", x_1)
# step 17
# T_prime subset of S denote the set of strings that have the values in dif_values on the bits dif_qubits
T_prime = [x for x in S if all(x[q] == v for q, v in zip(dif_qubits, dif_values))]
# Step 18: Remove x_1 from T'
T_prime.remove(x_1)
# Step 19: Second While loop for T_prime
while len(T_prime) > 1:
    # Step 22: Find the qubit b_prime
    R = find_qubit_with_unequal_sets(T_prime)  # Implement logic to find the best qubit
    b_prime = R[0]
    T_prime_0 = R[1]
    T_prime_1 = R[2]
    
    # Step 7: Append b to dif_qubits
    
    dif_qubits.append(b_prime)
    
    if len(T_prime_0) < len(T_prime_1):
        # Step 9: Set T = T_0 and append 0 to dif_values
        T_prime = T_prime_0
        dif_values.append(0)
    else:
        # Step 10: Set T = T_1 and append 1 to dif_values
        T_prime = T_prime_1
        dif_values.append(1)
x_2 = T_prime[0]
print("x2 is ",x_2)
from classiq import Model, synthesize, QReg,show
from classiq.builtin_functions import RZGate,StatePreparation,CXGate,XGate
model = Model()
probabilities = [0, 0, 0, 0,0.25 , 0.25, 0, 0.25,0,0,0.25,0,0,0,0,0]
sp = StatePreparation(
    probabilities=probabilities, error_metric={"KL": {"upper_bound": 0.1}}
)
t=model.StatePreparation(params=sp)
x_params= XGate()
cx_params=["cx1","cx2","cx3","cx4"]
cx_out=["co1","co2","co3","co4"]
l=1
l1=[1,1,1,1]
u=0
# u will be used for numbering of CX Gates
if x_1[dif] != 1:
    t_out_dif= model.XGate(x_params, in_wires={"TARGET":t["OUT"][dif]})
    l=0                                                                  # if l=0 then dif wire is t_out["TARGET"] otherwise t["OUT"][dif]
# Step 32: For qubit b in {1, 2, ..., n} \ dif
count=[1,1,1,1]
for b in range(0,4):
    if b != dif and x_1[b] != x_2[b]:
        count[b]=0
        cx_params[u] = CXGate()
        l1[b]=0
        # Step 34: Add a CNOT gate targeting b controlled on dif
        if l==0:
            cx_out[b]=model.CXGate(cx_params[u],in_wires={"TARGET":t["OUT"][b],"CTRL":t_out_dif["TARGET"]})            
        else:
            cx_out[b]=model.CXGate(cx_params[u],in_wires={"TARGET":t["OUT"][b],"CTRL":t["OUT"][dif]}) 
        u=u+1
        
x_last=["x1","x2","x3","x4"]
x_last_out=["xl1","xl2","xl3","xl4"]
# Step 37: For qubit b in dif_qubits
v=0
# v will be used for numbering of CX Gates
for b in dif_qubits:
    if x_2[b] != 1:
        x_last[i] = XGate()
        # Step 39: Add a NOT gate on line b
        if l1[b]==0:
            x_last_out[b]=model.XGate(x_last[v],in_wires={"TARGET":cx_out[b]["TARGET"]})
        else:
            x_last_out[b]=model.XGate(x_last[v],in_wires={"TARGET":t["OUT"][b]})
        v=v+1
dif_wire_number=7
for b in reversed(range(4)):
    if count[b]==0:
        dif_wire_number=b
        break

# now I" ll operate control G gate on dif qubit controlled by dif_qubits
# first I am creating a Gate G
import numpy as np
from classiq import Model, synthesize, show
from classiq.builtin_functions import UnitaryGate
alpha = 1/np.sqrt(2) # Define your own alpha
beta = 1/np.sqrt(2) # Define your own beta

num_gate_qubits = 1
gate_matrix = np.identity(2**num_gate_qubits, dtype=complex)
data_next = np.identity(2**num_gate_qubits, dtype=complex)
data_next[np.ix_([0,1],[0,1])] = [
        [
             alpha ,    beta                                                                                                                                                     
        ],
        [-beta, alpha]]
gate_matrix = np.dot(gate_matrix, data_next)
gate_matrix = gate_matrix.tolist()
unitary_gate_params = UnitaryGate(data=gate_matrix)
if dif_wire_number<4:
    G= model.UnitaryGate(unitary_gate_params, in_wires= {"TARGET":cx_out[dif_wire_number]["CTRL"]})
if dif_wire_number>4 and l==0:
    G= model.UnitaryGate(unitary_gate_params, in_wires= {"TARGET":t_out_dif["TARGET"]})
if dif_wire_number>4 and l==1:
    G= model.UnitaryGate(unitary_gate_params, in_wires= {"TARGET":t["OUT"][dif]})
    
quantum_program = synthesize(model.get_model())
show(quantum_program)    

This program is just to see whether we can genrate circuit just before merging step in algorithm 1 
 using Python SDK
 input S = [[0, 1, 0,1], [1,0, 0, 0], [0, 1,1, 0],[1,0,1,1]] 
dif is   2 
 i.e. our 3rd qubit line will act as dif since numbering started from 0 
x1 is   [1, 0, 1, 1]
x2 is  [1, 0, 0, 0]
Opening: https://platform.classiq.io/circuit/612ca234-7a77-4981-8229-a74cb2b05581?version=0.29.2
